下面我设计两组小数据：

1. **PCA 演示数据**：二维、4 个样本，用来完整手算 PCA：去均值 → 协方差矩阵 → 特征值/特征向量 → 主成分投影 → 白化。
2. **ICA 演示数据**：先设计两个独立源信号 $S$，再用混合矩阵 $A$ 混合成观测信号 $X$，然后演示 ICA：混合数据 → PCA 白化 → 旋转搜索 → 找回独立源。

为了计算整洁，下面统一使用 **总体协方差**：

$$
C = \frac{1}{N}XX^T
$$

而不是 `np.cov` 默认的：

$$
C = \frac{1}{N-1}XX^T
$$

这不影响 PCA/ICA 的核心理解，只是缩放系数不同。

---

# 一、PCA 演示数据设计

设计一个二维数据矩阵：

$$
X =
\begin{bmatrix}
2 & 0 & -2 & 0 \
1 & 1 & -1 & -1
\end{bmatrix}
$$

这里：

```text
每一列 = 一个样本
每一行 = 一个变量 / 一个 EEG 通道
```

所以 4 个样本分别是：

$$
x_1 =
\begin{bmatrix}
2 \
1
\end{bmatrix},
\quad
x_2 =
\begin{bmatrix}
0 \
1
\end{bmatrix},
\quad
x_3 =
\begin{bmatrix}
-2 \
-1
\end{bmatrix},
\quad
x_4 =
\begin{bmatrix}
0 \
-1
\end{bmatrix}
$$

---

# 二、PCA 第一步：检查均值

第一行均值：

$$
\bar{x}_1 = \frac{2 + 0 - 2 + 0}{4} = 0
$$

第二行均值：

$$
\bar{x}_2 = \frac{1 + 1 - 1 - 1}{4} = 0
$$

所以这个数据已经中心化：

$$
\bar{X} = 0
$$

不需要再减均值。

---

# 三、PCA 第二步：计算协方差矩阵

PCA 的协方差矩阵为：

$$
C = \frac{1}{N}XX^T
$$

这里 $N = 4$。

先计算：

$$
XX^T =
\begin{bmatrix}
2 & 0 & -2 & 0 \
1 & 1 & -1 & -1
\end{bmatrix}
\begin{bmatrix}
2 & 1 \
0 & 1 \
-2 & -1 \
0 & -1
\end{bmatrix}
$$

逐项算：

第一行第一列：

$$
2^2 + 0^2 + (-2)^2 + 0^2 = 4 + 0 + 4 + 0 = 8
$$

第一行第二列：

$$
2 \cdot 1 + 0 \cdot 1 + (-2)(-1) + 0(-1)
= 2 + 0 + 2 + 0 = 4
$$

第二行第一列：

$$
1 \cdot 2 + 1 \cdot 0 + (-1)(-2) + (-1) \cdot 0
= 2 + 0 + 2 + 0 = 4
$$

第二行第二列：

$$
1^2 + 1^2 + (-1)^2 + (-1)^2
= 1 + 1 + 1 + 1 = 4
$$

所以：

$$
XX^T =
\begin{bmatrix}
8 & 4 \
4 & 4
\end{bmatrix}
$$

因此：

$$
C =
\frac{1}{4}
\begin{bmatrix}
8 & 4 \
4 & 4
\end{bmatrix}
=

\begin{bmatrix}
2 & 1 \
1 & 1
\end{bmatrix}
$$

这个矩阵说明：

```text
第 1 个变量方差 = 2
第 2 个变量方差 = 1
二者协方差 = 1
```

也就是说两个变量不是独立方向，它们有线性相关性。

---

# 四、PCA 第三步：求协方差矩阵的特征值

要求：

$$
C v = \lambda v
$$

也就是：

$$
\det(C - \lambda I) = 0
$$

代入：

$$
C - \lambda I =
\begin{bmatrix}
2-\lambda & 1 \
1 & 1-\lambda
\end{bmatrix}
$$

行列式为：

$$
\det(C - \lambda I)
=

(2-\lambda)(1-\lambda) - 1
$$

展开：

$$
(2-\lambda)(1-\lambda) - 1
=

2 - 2\lambda - \lambda + \lambda^2 - 1
$$

所以：

$$
\lambda^2 - 3\lambda + 1 = 0
$$

解这个二次方程：

$$
\lambda =
\frac{3 \pm \sqrt{9 - 4}}{2}
=

\frac{3 \pm \sqrt{5}}{2}
$$

所以两个特征值是：

$$
\lambda_1 = \frac{3 + \sqrt{5}}{2} \approx 2.618
$$

$$
\lambda_2 = \frac{3 - \sqrt{5}}{2} \approx 0.382
$$

因为：

$$
\lambda_1 > \lambda_2
$$

所以第一主成分对应 $\lambda_1$，第二主成分对应 $\lambda_2$。

---

# 五、PCA 第四步：求第一主成分方向

对：

$$
\lambda_1 = \frac{3 + \sqrt{5}}{2}
$$

解：

$$
(C - \lambda_1 I)v_1 = 0
$$

即：

$$
\begin{bmatrix}
2-\lambda_1 & 1 \
1 & 1-\lambda_1
\end{bmatrix}
\begin{bmatrix}
a \
b
\end{bmatrix}
= 0
$$

看第一行：

$$
(2-\lambda_1)a + b = 0
$$

所以：

$$
b = (\lambda_1 - 2)a
$$

计算：

$$
\lambda_1 - 2
=

# \frac{3+\sqrt{5}}{2} - 2

\frac{\sqrt{5}-1}{2}
$$

而：

$$
\frac{\sqrt{5}-1}{2} \approx 0.618
$$

令 $a = 1$，则：

$$
b = 0.618
$$

所以第一主成分方向可以写成：

$$
v_1 =
\begin{bmatrix}
1 \
0.618
\end{bmatrix}
$$

但 PCA 通常使用单位向量，所以要归一化。

模长：

$$
|v_1|
=

\sqrt{1^2 + 0.618^2}
\approx
\sqrt{1 + 0.382}
=

\sqrt{1.382}
\approx 1.176
$$

所以单位向量：

$$
u_1 =
\frac{1}{1.176}
\begin{bmatrix}
1 \
0.618
\end{bmatrix}
\approx
\begin{bmatrix}
0.851 \
0.526
\end{bmatrix}
$$

这就是第一主成分方向。

---

# 六、PCA 第五步：求第二主成分方向

对：

$$
\lambda_2 = \frac{3 - \sqrt{5}}{2}
$$

同样有：

$$
(2-\lambda_2)a + b = 0
$$

所以：

$$
b = (\lambda_2 - 2)a
$$

计算：

$$
\lambda_2 - 2
=

# \frac{3-\sqrt{5}}{2} - 2

-\frac{1+\sqrt{5}}{2}
\approx -1.618
$$

令 $a = 1$，则：

$$
b = -1.618
$$

所以：

$$
v_2 =
\begin{bmatrix}
1 \
-1.618
\end{bmatrix}
$$

模长：

$$
|v_2|
=

\sqrt{1^2 + (-1.618)^2}
\approx
\sqrt{1 + 2.618}
=
\sqrt{3.618}
\approx 1.902
$$

单位化：

$$
u_2 =
\frac{1}{1.902}
\begin{bmatrix}
1 \
-1.618
\end{bmatrix}
\approx
\begin{bmatrix}
0.526 \
-0.851
\end{bmatrix}
$$

---

# 七、PCA 第六步：组成 PCA 变换矩阵

把两个单位特征向量作为列向量：

$$
U =
\begin{bmatrix}
0.851 & 0.526 \
0.526 & -0.851
\end{bmatrix}
$$

其中：

```text
第一列 u1：第一主成分方向
第二列 u2：第二主成分方向
```

这个矩阵是正交矩阵：

$$
U^T U = I
$$

也就是说两个主成分方向互相垂直。

---

# 八、PCA 第七步：把原始数据投影到主成分方向

PCA 投影为：

$$
Z = U^T X
$$

也就是：

$$
Z =
\begin{bmatrix}
0.851 & 0.526 \
0.526 & -0.851
\end{bmatrix}^T
\begin{bmatrix}
2 & 0 & -2 & 0 \
1 & 1 & -1 & -1
\end{bmatrix}
$$

因为这里 $U$ 是对称形式，所以 $U^T = U$，但一般情况下不要默认它们相等。

计算第一主成分得分：

第 1 个样本：

$$
z_{1,1}
=

# 0.851 \cdot 2 + 0.526 \cdot 1

# 1.702 + 0.526

2.228
$$

第 2 个样本：

$$
z_{1,2}
=

# 0.851 \cdot 0 + 0.526 \cdot 1

0.526
$$

第 3 个样本：

$$
z_{1,3}
=

# 0.851(-2) + 0.526(-1)

# -1.702 - 0.526

-2.228
$$

第 4 个样本：

$$
z_{1,4}
=

# 0.851 \cdot 0 + 0.526(-1)

-0.526
$$

所以第一主成分信号约为：

$$
PC_1 =
\begin{bmatrix}
2.228 & 0.526 & -2.228 & -0.526
\end{bmatrix}
$$

第二主成分得分：

第 1 个样本：

$$
z_{2,1}
=

# 0.526 \cdot 2 + (-0.851) \cdot 1

# 1.052 - 0.851

0.201
$$

第 2 个样本：

$$
z_{2,2}
=

# 0.526 \cdot 0 + (-0.851) \cdot 1

-0.851
$$

第 3 个样本：

$$
z_{2,3}
=

# 0.526(-2) + (-0.851)(-1)

# -1.052 + 0.851

-0.201
$$

第 4 个样本：

$$
z_{2,4}
=

# 0.526 \cdot 0 + (-0.851)(-1)

0.851
$$

所以：

$$
PC_2 =
\begin{bmatrix}
0.201 & -0.851 & -0.201 & 0.851
\end{bmatrix}
$$

因此：

$$
Z =
\begin{bmatrix}
2.228 & 0.526 & -2.228 & -0.526 \
0.201 & -0.851 & -0.201 & 0.851
\end{bmatrix}
$$

---

# 九、PCA 第八步：验证 PCA 后的协方差

PCA 后的协方差应该是对角矩阵：

$$
C_Z = \frac{1}{N}ZZ^T
$$

理论上：

$$
C_Z = U^T C U = \Lambda
$$

也就是：

$$
C_Z =
\begin{bmatrix}
2.618 & 0 \
0 & 0.382
\end{bmatrix}
$$

这说明：

```text
PCA 后两个成分不相关
第一主成分方差最大
第二主成分方差较小
```

注意：PCA 做到的是 **不相关**，不是统计独立。

---

# 十、PCA 第九步：白化

ICA 前经常会做 PCA 白化。

白化目标是：

$$
\operatorname{Cov}(X_{\text{white}}) = I
$$

PCA 后：

$$
Z = U^T X
$$

但是：

$$
\operatorname{Cov}(Z) =
\begin{bmatrix}
\lambda_1 & 0 \
0 & \lambda_2
\end{bmatrix}
$$

要让每个方向方差变成 1，需要除以标准差：

$$
X_{\text{white}}
=

\Lambda^{-1/2} U^T X
$$

其中：

$$
\Lambda^{-1/2}
=

\begin{bmatrix}
\frac{1}{\sqrt{\lambda_1}} & 0 \
0 & \frac{1}{\sqrt{\lambda_2}}
\end{bmatrix}
$$

因为：

$$
\lambda_1 \approx 2.618
$$

所以：

$$
\frac{1}{\sqrt{\lambda_1}}
\approx
\frac{1}{1.618}
\approx 0.618
$$

因为：

$$
\lambda_2 \approx 0.382
$$

所以：

$$
\frac{1}{\sqrt{\lambda_2}}
\approx
\frac{1}{0.618}
\approx 1.618
$$

因此：

$$
X_{\text{white}}
=

\begin{bmatrix}
0.618 & 0 \
0 & 1.618
\end{bmatrix}
Z
$$

白化后的协方差就是：

$$
\operatorname{Cov}(X_{\text{white}})
=

I
$$

---

# 十一、ICA 演示数据设计

现在设计 ICA 的数据。

我们先人为构造两个独立源：

$$
S =
\begin{bmatrix}
1 & -1 & 1 & -1 \
1 & 1 & -1 & -1
\end{bmatrix}
$$

其中：

```text
第一行 s1：源信号 1
第二行 s2：源信号 2
每一列：一个时间点 / 一个样本
```

也就是：

$$
s_1 =
\begin{bmatrix}
1 & -1 & 1 & -1
\end{bmatrix}
$$

$$
s_2 =
\begin{bmatrix}
1 & 1 & -1 & -1
\end{bmatrix}
$$

这两个源都是非高斯的。它们只取 $+1$ 和 $-1$，不是高斯分布。

---

# 十二、验证源信号的均值和协方差

第一行均值：

$$
\frac{1 + (-1) + 1 + (-1)}{4} = 0
$$

第二行均值：

$$
\frac{1 + 1 + (-1) + (-1)}{4} = 0
$$

所以：

$$
\bar{S} = 0
$$

计算：

$$
SS^T =
\begin{bmatrix}
1 & -1 & 1 & -1 \
1 & 1 & -1 & -1
\end{bmatrix}
\begin{bmatrix}
1 & 1 \
-1 & 1 \
1 & -1 \
-1 & -1
\end{bmatrix}
$$

第一行第一列：

$$
1^2 + (-1)^2 + 1^2 + (-1)^2 = 4
$$

第一行第二列：

$$
1 \cdot 1 + (-1) \cdot 1 + 1 \cdot (-1) + (-1)(-1)
=

1 - 1 - 1 + 1 = 0
$$

第二行第一列同样是 0。

第二行第二列：

$$
1^2 + 1^2 + (-1)^2 + (-1)^2 = 4
$$

所以：

$$
SS^T =
\begin{bmatrix}
4 & 0 \
0 & 4
\end{bmatrix}
$$

因此：

$$
\operatorname{Cov}(S)
=

# \frac{1}{4}SS^T

\begin{bmatrix}
1 & 0 \
0 & 1
\end{bmatrix}
=

I
$$

说明两个源信号：

```text
均值为 0
方差为 1
协方差为 0
```

而且它们还不仅仅是不相关，而是人为设计成独立的离散源。

---

# 十三、用混合矩阵生成观测 EEG 信号

设计混合矩阵：

$$
A =
\begin{bmatrix}
2 & 1 \
1 & 2
\end{bmatrix}
$$

观测信号为：

$$
X = AS
$$

计算：

$$
X =
\begin{bmatrix}
2 & 1 \
1 & 2
\end{bmatrix}
\begin{bmatrix}
1 & -1 & 1 & -1 \
1 & 1 & -1 & -1
\end{bmatrix}
$$

逐列算：

第 1 列：

$$
x_1 =
\begin{bmatrix}
2 & 1 \
1 & 2
\end{bmatrix}
\begin{bmatrix}
1 \
1
\end{bmatrix}
=

\begin{bmatrix}
3 \
3
\end{bmatrix}
$$

第 2 列：

$$
x_2 =
\begin{bmatrix}
2 & 1 \
1 & 2
\end{bmatrix}
\begin{bmatrix}
-1 \
1
\end{bmatrix}
=

\begin{bmatrix}
-2 + 1 \
-1 + 2
\end{bmatrix}
=

\begin{bmatrix}
-1 \
1
\end{bmatrix}
$$

第 3 列：

$$
x_3 =
\begin{bmatrix}
2 & 1 \
1 & 2
\end{bmatrix}
\begin{bmatrix}
1 \
-1
\end{bmatrix}
=

\begin{bmatrix}
2 - 1 \
1 - 2
\end{bmatrix}
=

\begin{bmatrix}
1 \
-1
\end{bmatrix}
$$

第 4 列：

$$
x_4 =
\begin{bmatrix}
2 & 1 \
1 & 2
\end{bmatrix}
\begin{bmatrix}
-1 \
-1
\end{bmatrix}
=

\begin{bmatrix}
-3 \
-3
\end{bmatrix}
$$

所以：

$$
X =
\begin{bmatrix}
3 & -1 & 1 & -3 \
3 & 1 & -1 & -3
\end{bmatrix}
$$

这个 $X$ 就像 EEG 里的观测通道信号。

也就是：

```text
两个真实源 S，被混合矩阵 A 混合后，变成两个观测通道 X
```

ICA 的任务就是：

$$
X \longrightarrow S
$$

也就是从混合信号中找回源信号。

---

# 十四、ICA 第一步：计算混合信号协方差

因为：

$$
X = AS
$$

并且：

$$
\operatorname{Cov}(S) = I
$$

所以：

$$
\operatorname{Cov}(X)
=

# A \operatorname{Cov}(S) A^T

AA^T
$$

计算：

$$
AA^T =
\begin{bmatrix}
2 & 1 \
1 & 2
\end{bmatrix}
\begin{bmatrix}
2 & 1 \
1 & 2
\end{bmatrix}
$$

第一行第一列：

$$
2^2 + 1^2 = 5
$$

第一行第二列：

$$
2 \cdot 1 + 1 \cdot 2 = 4
$$

第二行第一列：

$$
1 \cdot 2 + 2 \cdot 1 = 4
$$

第二行第二列：

$$
1^2 + 2^2 = 5
$$

所以：

$$
C_X =
\operatorname{Cov}(X)
=

\begin{bmatrix}
5 & 4 \
4 & 5
\end{bmatrix}
$$

这说明两个观测通道高度相关。

---

# 十五、ICA 第二步：先 PCA 白化

对：

$$
C_X =
\begin{bmatrix}
5 & 4 \
4 & 5
\end{bmatrix}
$$

求特征值。

解：

$$
\det(C_X - \lambda I) = 0
$$

即：

$$
\det
\begin{bmatrix}
5-\lambda & 4 \
4 & 5-\lambda
\end{bmatrix}
= 0
$$

所以：

$$
(5-\lambda)^2 - 16 = 0
$$

$$
(5-\lambda)^2 = 16
$$

$$
5-\lambda = \pm 4
$$

因此：

$$
\lambda_1 = 9
$$

$$
\lambda_2 = 1
$$

---

## 1. 求 $\lambda_1 = 9$ 的特征向量

解：

$$
(C_X - 9I)u_1 = 0
$$

$$
\begin{bmatrix}
-4 & 4 \
4 & -4
\end{bmatrix}
\begin{bmatrix}
a \
b
\end{bmatrix}
= 0
$$

第一行：

$$
-4a + 4b = 0
$$

所以：

$$
a = b
$$

取：

$$
u_1 =
\frac{1}{\sqrt{2}}
\begin{bmatrix}
1 \
1
\end{bmatrix}
$$

---

## 2. 求 $\lambda_2 = 1$ 的特征向量

解：

$$
(C_X - I)u_2 = 0
$$

$$
\begin{bmatrix}
4 & 4 \
4 & 4
\end{bmatrix}
\begin{bmatrix}
a \
b
\end{bmatrix}
= 0
$$

所以：

$$
4a + 4b = 0
$$

$$
a = -b
$$

取：

$$
u_2 =
\frac{1}{\sqrt{2}}
\begin{bmatrix}
1 \
-1
\end{bmatrix}
$$

因此：

$$
U =
\frac{1}{\sqrt{2}}
\begin{bmatrix}
1 & 1 \
1 & -1
\end{bmatrix}
$$

并且：

$$
\Lambda =
\begin{bmatrix}
9 & 0 \
0 & 1
\end{bmatrix}
$$

---

## 3. 构造白化矩阵

白化矩阵：

$$
V =
\Lambda^{-1/2}U^T
$$

其中：

$$
\Lambda^{-1/2}
=

\begin{bmatrix}
\frac{1}{3} & 0 \
0 & 1
\end{bmatrix}
$$

所以：

$$
V =
\begin{bmatrix}
\frac{1}{3} & 0 \
0 & 1
\end{bmatrix}
\frac{1}{\sqrt{2}}
\begin{bmatrix}
1 & 1 \
1 & -1
\end{bmatrix}
$$

---

# 十六、ICA 第三步：对白化矩阵作用到 X

白化后的数据：

$$
Z = VX = \Lambda^{-1/2}U^TX
$$

先计算：

$$
U^TX
=

\frac{1}{\sqrt{2}}
\begin{bmatrix}
1 & 1 \
1 & -1
\end{bmatrix}
\begin{bmatrix}
3 & -1 & 1 & -3 \
3 & 1 & -1 & -3
\end{bmatrix}
$$

第一行：

$$
\frac{1}{\sqrt{2}}(X_1 + X_2)
$$

也就是：

$$
\frac{1}{\sqrt{2}}
\begin{bmatrix}
3+3 & -1+1 & 1+(-1) & -3+(-3)
\end{bmatrix}
$$

# $$

\frac{1}{\sqrt{2}}
\begin{bmatrix}
6 & 0 & 0 & -6
\end{bmatrix}
$$

# $$

\begin{bmatrix}
3\sqrt{2} & 0 & 0 & -3\sqrt{2}
\end{bmatrix}
$$

第二行：

$$
\frac{1}{\sqrt{2}}(X_1 - X_2)
$$

即：

$$
\frac{1}{\sqrt{2}}
\begin{bmatrix}
3-3 & -1-1 & 1-(-1) & -3-(-3)
\end{bmatrix}
$$

# $$

\frac{1}{\sqrt{2}}
\begin{bmatrix}
0 & -2 & 2 & 0
\end{bmatrix}
$$

# $$

\begin{bmatrix}
0 & -\sqrt{2} & \sqrt{2} & 0
\end{bmatrix}
$$

所以：

$$
U^TX =
\begin{bmatrix}
3\sqrt{2} & 0 & 0 & -3\sqrt{2} \
0 & -\sqrt{2} & \sqrt{2} & 0
\end{bmatrix}
$$

再乘：

$$
\Lambda^{-1/2}
=

\begin{bmatrix}
\frac{1}{3} & 0 \
0 & 1
\end{bmatrix}
$$

得到：

$$
Z =
\begin{bmatrix}
\sqrt{2} & 0 & 0 & -\sqrt{2} \
0 & -\sqrt{2} & \sqrt{2} & 0
\end{bmatrix}
$$

验证：

$$
\operatorname{Cov}(Z) = I
$$

所以现在数据已经白化。

---

# 十七、白化后的意义

白化前：

$$
X = AS
$$

其中 $A$ 是一般矩阵。

白化后：

$$
Z = VX = VAS
$$

由于白化使协方差变成单位矩阵，所以 $VA$ 变成一个正交变换，也就是旋转或反射。

这意味着：

```text
白化之后，ICA 不需要再找任意矩阵，
只需要找一个旋转方向，把 Z 旋转回独立源 S。
```





# 十八、ICA 第四步：搜索旋转方向

二维情况下，任意单位方向可以写成：

$$
w(\theta) =
\begin{bmatrix}
\cos\theta \
\sin\theta
\end{bmatrix}
$$

把白化数据投影到这个方向：

$$
y(\theta) = w(\theta)^T Z
$$

代入：

$$
Z =
\begin{bmatrix}
\sqrt{2} & 0 & 0 & -\sqrt{2} \
0 & -\sqrt{2} & \sqrt{2} & 0
\end{bmatrix}
$$

所以：

$$
y(\theta)
=

\begin{bmatrix}
\cos\theta & \sin\theta
\end{bmatrix}
\begin{bmatrix}
\sqrt{2} & 0 & 0 & -\sqrt{2} \
0 & -\sqrt{2} & \sqrt{2} & 0
\end{bmatrix}
$$

逐项得到：

$$
y(\theta)
=

\begin{bmatrix}
\sqrt{2}\cos\theta,
-\sqrt{2}\sin\theta,
\sqrt{2}\sin\theta,
-\sqrt{2}\cos\theta
\end{bmatrix}
$$

ICA 要找的是：哪个 $\theta$ 使得 $y(\theta)$ 最非高斯。

---

# 十九、用四阶矩 / 峰度衡量非高斯性

因为白化后：

$$
E[y^2] = 1
$$

可以计算四阶矩：

$$
E[y^4]
=

\frac{1}{4}
\left[
(\sqrt{2}\cos\theta)^4
+
(-\sqrt{2}\sin\theta)^4
+
(\sqrt{2}\sin\theta)^4
+
(-\sqrt{2}\cos\theta)^4
\right]
$$

分别算：

$$
(\sqrt{2}\cos\theta)^4 = 4\cos^4\theta
$$

$$
(-\sqrt{2}\sin\theta)^4 = 4\sin^4\theta
$$

所以：

$$
E[y^4]
=

\frac{1}{4}
\left[
4\cos^4\theta
+
4\sin^4\theta
+
4\sin^4\theta
+
4\cos^4\theta
\right]
$$

# $$

\frac{1}{4}
\left[
8\cos^4\theta + 8\sin^4\theta
\right]
$$

# $$

2(\cos^4\theta + \sin^4\theta)
$$

峰度 excess kurtosis：

$$
\kappa(\theta)
=

E[y^4] - 3
$$

所以：

$$
\kappa(\theta)
=

2(\cos^4\theta + \sin^4\theta) - 3
$$

利用：

$$
\cos^4\theta + \sin^4\theta
=

(\cos^2\theta + \sin^2\theta)^2 - 2\cos^2\theta\sin^2\theta
$$

因为：

$$
\cos^2\theta + \sin^2\theta = 1
$$

所以：

$$
\cos^4\theta + \sin^4\theta
=

1 - 2\cos^2\theta\sin^2\theta
$$

因此：

$$
\kappa(\theta)
=

2(1 - 2\cos^2\theta\sin^2\theta) - 3
$$

# $$

-1 - 4\cos^2\theta\sin^2\theta
$$

ICA 常用非高斯性大小，即：

$$
|\kappa(\theta)|
$$

要让：

$$
|\kappa(\theta)|
$$

最大，就要让：

$$
\cos^2\theta\sin^2\theta
$$

最大。

因为：

$$
\cos^2\theta + \sin^2\theta = 1
$$

所以当：

$$
\cos^2\theta = \sin^2\theta = \frac{1}{2}
$$

时乘积最大。

因此：

$$
\theta = 45^\circ
$$

也就是：

$$
w_1 =
\begin{bmatrix}
\frac{1}{\sqrt{2}} \
\frac{1}{\sqrt{2}}
\end{bmatrix}
$$





# 二十、用这个方向恢复第一个独立源

令：

$$
w_1 =
\begin{bmatrix}
\frac{1}{\sqrt{2}} \
\frac{1}{\sqrt{2}}
\end{bmatrix}
$$

计算：

$$
y_1 = w_1^TZ
$$

即：

$$
y_1 =
\begin{bmatrix}
\frac{1}{\sqrt{2}} & \frac{1}{\sqrt{2}}
\end{bmatrix}
\begin{bmatrix}
\sqrt{2} & 0 & 0 & -\sqrt{2} \
0 & -\sqrt{2} & \sqrt{2} & 0
\end{bmatrix}
$$

逐项：

第 1 项：

$$
\frac{1}{\sqrt{2}}\sqrt{2} + \frac{1}{\sqrt{2}}0 = 1
$$

第 2 项：

$$
\frac{1}{\sqrt{2}}0 + \frac{1}{\sqrt{2}}(-\sqrt{2}) = -1
$$

第 3 项：

$$
\frac{1}{\sqrt{2}}0 + \frac{1}{\sqrt{2}}\sqrt{2} = 1
$$

第 4 项：

$$
\frac{1}{\sqrt{2}}(-\sqrt{2}) + \frac{1}{\sqrt{2}}0 = -1
$$

所以：

$$
y_1 =
\begin{bmatrix}
1 & -1 & 1 & -1
\end{bmatrix}
$$

这正好等于：

$$
s_1
$$

---

# 二十一、恢复第二个独立源

第二个方向必须和第一个方向正交：

$$
w_2 =
\begin{bmatrix}
\frac{1}{\sqrt{2}} \
-\frac{1}{\sqrt{2}}
\end{bmatrix}
$$

计算：

$$
y_2 = w_2^TZ
$$

即：

$$
y_2 =
\begin{bmatrix}
\frac{1}{\sqrt{2}} & -\frac{1}{\sqrt{2}}
\end{bmatrix}
\begin{bmatrix}
\sqrt{2} & 0 & 0 & -\sqrt{2} \
0 & -\sqrt{2} & \sqrt{2} & 0
\end{bmatrix}
$$

逐项：

第 1 项：

$$
1 + 0 = 1
$$

第 2 项：

$$
0 + 1 = 1
$$

第 3 项：

$$
0 - 1 = -1
$$

第 4 项：

$$
-1 + 0 = -1
$$

所以：

$$
y_2 =
\begin{bmatrix}
1 & 1 & -1 & -1
\end{bmatrix}
$$

这正好等于：

$$
s_2
$$

因此：

$$
Y =
\begin{bmatrix}
y_1 \
y_2
\end{bmatrix}
=

\begin{bmatrix}
1 & -1 & 1 & -1 \
1 & 1 & -1 & -1
\end{bmatrix}
=

S
$$

ICA 成功恢复了源信号。





# 二十二、PCA 和 ICA 在这个例子中的关系

完整过程是：

```text
真实独立源 S
  ↓ 经过混合矩阵 A
观测信号 X = A S
  ↓ PCA
找到最大方差方向
  ↓ 白化
把数据变成 Cov = I 的球状分布
  ↓ ICA
在白化空间中旋转，找最非高斯方向
  ↓
恢复独立源 S
```

PCA 只做到：

$$
\operatorname{Cov}(Z) =
\begin{bmatrix}
\lambda_1 & 0 \
0 & \lambda_2
\end{bmatrix}
$$

或者白化后：

$$
\operatorname{Cov}(Z_{\text{white}}) = I
$$

但 PCA 不保证源信号独立。

ICA 在 PCA 白化基础上继续做：

$$
S \approx W X
$$

其中：

$$
W = R V
$$

这里：

```text
V：PCA 白化矩阵
R：ICA 找到的旋转矩阵
W：最终解混矩阵
```

在这个例子中：

$$
R =
\begin{bmatrix}
\frac{1}{\sqrt{2}} & \frac{1}{\sqrt{2}} \
\frac{1}{\sqrt{2}} & -\frac{1}{\sqrt{2}}
\end{bmatrix}
$$

所以：

$$
Y = RZ = RVX
$$

最终：

$$
Y = S
$$





# 二十三、完整 Python 代码：PCA 手算复现


In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# =========================
# PCA 演示数据
# =========================

X = np.array([
    [2, 0, -2, 0],
    [1, 1, -1, -1]
], dtype=float)

N = X.shape[1]

print("原始数据 X:")
print(X)

# 1. 均值
mean = X.mean(axis=1, keepdims=True)
print("\n每个变量的均值:")
print(mean)

X_centered = X - mean
print("\n中心化后的 X:")
print(X_centered)

# 2. 协方差矩阵，使用 1/N
C = (X_centered @ X_centered.T) / N
print("\n协方差矩阵 C = 1/N * X X^T:")
print(C)

# 3. 特征值分解
eigvals, eigvecs = np.linalg.eigh(C)

# eigh 返回升序，这里改成降序
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print("\n特征值，从大到小:")
print(eigvals)

print("\n特征向量，每一列对应一个主成分方向:")
print(eigvecs)

# 4. PCA 投影
Z = eigvecs.T @ X_centered
print("\nPCA 投影后的数据 Z = U^T X:")
print(Z)

# 5. 验证 PCA 后协方差
C_Z = (Z @ Z.T) / N
print("\nPCA 后的协方差:")
print(C_Z)

# 6. 白化
Lambda_inv_sqrt = np.diag(1.0 / np.sqrt(eigvals))
X_white = Lambda_inv_sqrt @ eigvecs.T @ X_centered

print("\n白化后的数据 X_white:")
print(X_white)

C_white = (X_white @ X_white.T) / N
print("\n白化后的协方差:")
print(C_white)




你会看到类似结果：

```text
协方差矩阵 C:
[[2. 1.]
 [1. 1.]]

特征值:
[2.618034 0.381966]

特征向量:
[[-0.850651  0.525731]
 [-0.525731 -0.850651]]

PCA 后协方差:
[[2.618034 0.      ]
 [0.       0.381966]]

白化后协方差:
[[1. 0.]
 [0. 1.]]
```

注意：特征向量的正负号可能和我手算相反，例如：

$$
u
$$

和：

$$
-u
$$

都是合法特征向量。

所以你看到正负号不同是正常的。





# 二十四、完整 Python 代码：ICA 手算复现


In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# =========================
# ICA 演示数据
# =========================

# 真实独立源 S
S = np.array([
    [1, -1, 1, -1],
    [1, 1, -1, -1]
], dtype=float)

N = S.shape[1]

print("真实源信号 S:")
print(S)

# 验证源信号协方差
C_S = (S @ S.T) / N
print("\n源信号协方差 Cov(S):")
print(C_S)

# 混合矩阵 A
A = np.array([
    [2, 1],
    [1, 2]
], dtype=float)

print("\n混合矩阵 A:")
print(A)

# 观测信号 X = A S
X = A @ S
print("\n观测信号 X = A S:")
print(X)

# 观测信号协方差
C_X = (X @ X.T) / N
print("\n观测信号协方差 Cov(X):")
print(C_X)

# =========================
# PCA 白化
# =========================

eigvals, eigvecs = np.linalg.eigh(C_X)

# 从大到小排序
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print("\nCov(X) 的特征值:")
print(eigvals)

print("\nCov(X) 的特征向量 U:")
print(eigvecs)

# 白化矩阵 V = Lambda^{-1/2} U^T
Lambda_inv_sqrt = np.diag(1.0 / np.sqrt(eigvals))
V = Lambda_inv_sqrt @ eigvecs.T

print("\n白化矩阵 V = Lambda^{-1/2} U^T:")
print(V)

# 白化后的数据 Z
Z = V @ X
print("\n白化后的数据 Z = V X:")
print(Z)

C_Z = (Z @ Z.T) / N
print("\n白化后 Cov(Z):")
print(C_Z)

# =========================
# ICA 旋转搜索
# =========================

# 构造角度搜索
thetas = np.linspace(0, np.pi, 361)

best_theta = None
best_score = -np.inf

scores = []

for theta in thetas:
    w = np.array([
        [np.cos(theta)],
        [np.sin(theta)]
    ])

    y = (w.T @ Z).ravel()

    # 因为 Z 已经白化，理论上 Var(y)=1
    # 使用 excess kurtosis = E[y^4] - 3(E[y^2])^2
    Ey2 = np.mean(y ** 2)
    Ey4 = np.mean(y ** 4)
    kurtosis_excess = Ey4 - 3 * (Ey2 ** 2)

    score = abs(kurtosis_excess)
    scores.append(score)

    if score > best_score:
        best_score = score
        best_theta = theta

print("\n搜索得到的最佳 theta，弧度:")
print(best_theta)

print("\n搜索得到的最佳 theta，角度:")
print(np.degrees(best_theta))

print("\n最大非高斯性分数 |excess kurtosis|:")
print(best_score)

# 使用理论上找到的两个 ICA 方向
R = np.array([
    [1 / np.sqrt(2), 1 / np.sqrt(2)],
    [1 / np.sqrt(2), -1 / np.sqrt(2)]
])

print("\nICA 旋转矩阵 R:")
print(R)

# 恢复源
Y = R @ Z

print("\nICA 恢复的信号 Y = R Z:")
print(Y)

print("\n原始真实源 S:")
print(S)

# 最终解混矩阵 W = R V
W = R @ V
print("\n最终解混矩阵 W = R V:")
print(W)

print("\n检查 W X:")
print(W @ X)



# 二十五、代码中的核心线性代数对应关系

## PCA 部分

```text
C = (X @ X.T) / N
```

对应：

$$
C = \frac{1}{N}XX^T
$$



```text
eigvals, eigvecs = np.linalg.eigh(C)
```

对应：

$$
C = U\Lambda U^T
$$



```text
Z = eigvecs.T @ X
```

对应：

$$
Z = U^T X
$$



```text
X_white = Lambda_inv_sqrt @ eigvecs.T @ X
```

对应：

$$
X_{\text{white}} = \Lambda^{-1/2}U^TX
$$



## ICA 部分

```text
X = A @ S
```

对应：

$$
X = AS
$$



```text
V = Lambda_inv_sqrt @ eigvecs.T
Z = V @ X
```

对应：

$$
Z = VX
$$



```text
R = np.array([
    [1 / np.sqrt(2), 1 / np.sqrt(2)],
    [1 / np.sqrt(2), -1 / np.sqrt(2)]
])
Y = R @ Z
```

对应：

$$
Y = RZ
$$

也就是：

$$
Y = RVX
$$

其中：

$$
W = RV
$$

所以：

$$
Y = WX
$$

这就是 ICA 的解混过程。





# 二十六、和 EEG ICA 的对应关系

在真实 EEG 中：

$$
X = AS
$$

可以理解为：

```text
X：头皮电极记录到的多通道 EEG
S：潜在独立源，比如脑源、眨眼、眼动、肌电、心电
A：头部体积传导和电极混合关系
```

PCA 在 ICA 里常常负责：

```text
1. 去相关
2. 白化
3. 降维
4. 简化搜索空间
```

ICA 负责：

```text
在白化后的空间里继续旋转，找最非高斯、最独立的方向
```

所以可以总结为：

```text
PCA：把混合数据整理成不相关、单位方差的空间
ICA：在这个空间里找真正的独立源方向
```

也就是：

$$
X
\overset{\text{PCA whitening}}{\longrightarrow}
Z
\overset{\text{ICA rotation}}{\longrightarrow}
S
$$

真实 MNE 里的 `ICA.fit(raw)` 大致也包含这个思想：

```text
Raw EEG
  ↓
中心化
  ↓
PCA / 白化 / 可能降维
  ↓
FastICA / Picard / Infomax 等算法寻找独立成分
  ↓
得到 ICA components
```

所以你看到的 `ICA000`, `ICA001`, `ICA002` 不是 PCA 主成分，而是：

```text
PCA 白化之后，再经过 ICA 旋转得到的独立成分
```
